# Apache Polaris — API Dependency Testing

**Date:** 2026-05-15  
**Reference:** https://polaris.apache.org/releases/1.3.0/polaris-api-specs/

## 목표
- 각 엔티티의 생성/삭제 시 의존성 동작 확인
- Polaris가 의존성 있는 삭제를 block하는지 cascade하는지 검증
- 각 에러 케이스의 HTTP status code 및 response format 확인
- 모든 테스트는 독립적 — 각 그룹이 자체 리소스 생성/삭제 (try/finally)

## 테스트 구조
```
Group 1: Authentication edge cases
Group 2: Principal Role dependencies
Group 3: Principal dependencies
Group 4: Catalog dependencies
Group 5: Catalog Role dependencies
Group 6: Namespace dependencies
Group 7: Table dependencies
Group 8: Full lifecycle sequential flow
Group 9: Error response format
```

In [293]:
# ── src path bootstrap + shared config (modules live in /src) ───────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from polaris_test_utils import *
init_env("local")   # default safe env; POLARIS_URL/REALM/ROOT_SECRET/MINIO_*/PG_* now set
# ============================================================
# Setup
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display

POLARIS_URL = POLARIS_URL  # from init_env (src/config)
REALM       = REALM  # from init_env
BASE        = f'{POLARIS_URL}/api/management/v1'
CAT         = f'{POLARIS_URL}/api/catalog/v1'

def get_token(client_id='root', client_secret=ROOT_SECRET, realm=REALM):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': realm},
        data={'grant_type': 'client_credentials', 'client_id': client_id,
              'client_secret': client_secret, 'scope': 'PRINCIPAL_ROLE:ALL'}
    )
    return r

def h(token, realm=REALM):
    return {'Authorization': f'Bearer {token}', 'Polaris-Realm': realm,
            'Content-Type': 'application/json'}

def safe_delete(url, token=None):
    """Delete silently — ignore all errors"""
    try:
        requests.delete(url, headers=h(token or TOKEN_ROOT))
    except:
        pass

RESULTS = []

def test(group, name, r, expected_status, note=''):
    actual = r.status_code
    passed = actual == expected_status
    icon   = '✅' if passed else '❌'
    RESULTS.append({'group': group, 'test': name,
                    'expected': expected_status, 'actual': actual,
                    'passed': passed, 'note': note})
    err = ''
    if r.text:
        try: err = r.json().get('error', {}).get('message', '')[:80]
        except: err = r.text[:80]
    print(f'  {icon} [{actual}] {name}')
    if not passed and err: print(f'       → {err}')
    if note: print(f'       💡 {note}')
    return r

# Verify root
r = get_token()
if r.status_code != 200:
    print(f'❌ Root auth failed: {r.json()}')
else:
    TOKEN_ROOT = r.json()['access_token']
    r2 = requests.get(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT))
    if r2.status_code == 403:
        print('❌ Root missing SERVICE_MANAGE_ACCESS — fix grants first')
    else:
        print(f'✅ Root verified — {len(r2.json().get("roles", []))} principal roles')
        print('   Ready to run tests')

✅ Root verified — 1 principal roles
   Ready to run tests


## Group 1 — Authentication Edge Cases

In [294]:
print('=== Group 1: Authentication Edge Cases ===')

r = requests.post(f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
    headers={'Polaris-Realm': 'WRONG-REALM'},
    data={'grant_type': 'client_credentials', 'client_id': 'root',
          'client_secret': ROOT_SECRET, 'scope': 'PRINCIPAL_ROLE:ALL'})
test('Auth', 'Wrong realm header', r, 404)

r = requests.post(f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
    headers={'Polaris-Realm': REALM},
    data={'grant_type': 'client_credentials', 'client_id': 'root',
          'client_secret': 'WRONG-SECRET', 'scope': 'PRINCIPAL_ROLE:ALL'})
test('Auth', 'Wrong client_secret', r, 401)

r = requests.post(f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
    headers={'Polaris-Realm': REALM},
    data={'grant_type': 'client_credentials', 'client_id': 'nonexistent',
          'client_secret': ROOT_SECRET, 'scope': 'PRINCIPAL_ROLE:ALL'})
test('Auth', 'Wrong client_id', r, 401)

r = requests.post(f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
    headers={'Content-Type': 'application/x-www-form-urlencoded'},
    data={'grant_type': 'client_credentials', 'client_id': 'root',
          'client_secret': ROOT_SECRET, 'scope': 'PRINCIPAL_ROLE:ALL'})
test('Auth', 'No realm header (requireHeader=true)', r, 404)

r = get_token()
test('Auth', 'Correct credentials', r, 200)
TOKEN_ROOT = r.json()['access_token']

r = requests.get(f'{CAT}/nonexistent-catalog/namespaces', headers=h(TOKEN_ROOT))
test('Auth', 'Root token on Catalog API (missing catalog)', r, 404,
     'FINDING: Root CAN call Catalog API — 404 for missing catalog')

r = requests.get(f'{BASE}/principals',
    headers={'Authorization': 'Bearer INVALID-TOKEN', 'Polaris-Realm': REALM})
test('Auth', 'Invalid token format', r, 401)

r = requests.get(f'{BASE}/principals', headers={'Polaris-Realm': REALM})
test('Auth', 'Missing Authorization header', r, 401)

=== Group 1: Authentication Edge Cases ===
  ✅ [404] Wrong realm header
  ✅ [401] Wrong client_secret
  ✅ [401] Wrong client_id
  ✅ [404] No realm header (requireHeader=true)
  ✅ [200] Correct credentials
  ✅ [404] Root token on Catalog API (missing catalog)
       💡 FINDING: Root CAN call Catalog API — 404 for missing catalog
  ✅ [401] Invalid token format
  ✅ [401] Missing Authorization header


<Response [401]>

## Group 2 — Principal Role Dependencies

In [295]:
print('=== Group 2: Principal Role Dependencies ===')

safe_delete(f'{BASE}/principals/test-principal-pr')
safe_delete(f'{BASE}/principal-roles/test-pr')
safe_delete(f'{BASE}/principal-roles/dep-test-pr')

try:
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'test-pr'}})
    requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'test-principal-pr', 'type': 'SERVICE'})
    requests.put(f'{BASE}/principals/test-principal-pr/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'test-pr'}})

    r = requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'dep-test-pr'}})
    test('PrincipalRole', 'Create principal role', r, 201)

    r = requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'dep-test-pr'}})
    test('PrincipalRole', 'Create duplicate principal role', r, 409)

    r = requests.delete(f'{BASE}/principal-roles/test-pr', headers=h(TOKEN_ROOT))
    test('PrincipalRole', 'Delete role with principal assigned', r, 204,
         'FINDING: CASCADE — principal survives, assignment removed')

    r = requests.get(f'{BASE}/principals/test-principal-pr', headers=h(TOKEN_ROOT))
    test('PrincipalRole', 'Principal exists after role deleted', r, 200,
         'FINDING: Principal survives role deletion')

    r = requests.get(f'{BASE}/principal-roles/test-pr', headers=h(TOKEN_ROOT))
    test('PrincipalRole', 'Get deleted principal role', r, 404)

    r = requests.delete(f'{BASE}/principal-roles/nonexistent-role', headers=h(TOKEN_ROOT))
    test('PrincipalRole', 'Delete non-existent principal role', r, 404)

finally:
    print('\n  🧹 Cleanup Group 2...')
    safe_delete(f'{BASE}/principals/test-principal-pr')
    safe_delete(f'{BASE}/principal-roles/test-pr')
    safe_delete(f'{BASE}/principal-roles/dep-test-pr')
    print('  🧹 Done')

=== Group 2: Principal Role Dependencies ===
  ✅ [201] Create principal role
  ✅ [409] Create duplicate principal role
  ✅ [204] Delete role with principal assigned
       💡 FINDING: CASCADE — principal survives, assignment removed
  ✅ [200] Principal exists after role deleted
       💡 FINDING: Principal survives role deletion
  ✅ [404] Get deleted principal role
  ✅ [404] Delete non-existent principal role

  🧹 Cleanup Group 2...
  🧹 Done


## Group 3 — Principal Dependencies

In [296]:
print('=== Group 3: Principal Dependencies ===')

safe_delete(f'{BASE}/principals/dep-test-user')
safe_delete(f'{BASE}/principal-roles/test-pr-for-principal')

TEST_CID = ''
TEST_CSECRET = ''

try:
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'test-pr-for-principal'}})

    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'dep-test-user', 'type': 'SERVICE'})
    test('Principal', 'Create principal', r, 201)
    creds = r.json().get('credentials', {})
    TEST_CID     = creds.get('clientId', '')
    TEST_CSECRET = creds.get('clientSecret', '')

    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'dep-test-user', 'type': 'SERVICE'})
    test('Principal', 'Create duplicate principal', r, 409)

    r = requests.put(f'{BASE}/principals/dep-test-user/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'nonexistent-role'}})
    test('Principal', 'Assign non-existent principal role', r, 404)

    r = requests.put(f'{BASE}/principals/dep-test-user/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'test-pr-for-principal'}})
    test('Principal', 'Assign valid principal role', r, 201)

    r = requests.put(f'{BASE}/principals/dep-test-user/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'test-pr-for-principal'}})
    test('Principal', 'Assign same principal role again', r, 500,
         'FINDING: 500 duplicate key — not idempotent')

    if TEST_CID and TEST_CSECRET:
        r_tok = get_token(TEST_CID, TEST_CSECRET)
        if r_tok.status_code == 200:
            user_token = r_tok.json()['access_token']
            r = requests.post(f'{BASE}/principals/dep-test-user/rotate',
                headers=h(user_token))
            test('Principal', 'Self-rotate credentials', r, 200)

    r = requests.post(f'{BASE}/principals/dep-test-user/rotate', headers=h(TOKEN_ROOT))
    test('Principal', 'Root rotate other principal', r, 403,
         'FINDING: Admin cannot rotate others — by design')

    r = requests.delete(f'{BASE}/principals/dep-test-user', headers=h(TOKEN_ROOT))
    test('Principal', 'Delete principal with role assigned', r, 204,
         'FINDING: CASCADE — role assignment removed')

    r = requests.get(f'{BASE}/principal-roles/test-pr-for-principal', headers=h(TOKEN_ROOT))
    test('PrincipalRole', 'Get principal role', r, 200,
         'FINDING: Principal Role not deleted')

    r = requests.get(f'{BASE}/principals/dep-test-user', headers=h(TOKEN_ROOT))
    test('Principal', 'Get deleted principal', r, 404)

finally:
    print('\n  🧹 Cleanup Group 3...')
    safe_delete(f'{BASE}/principals/dep-test-user')
    safe_delete(f'{BASE}/principal-roles/test-pr-for-principal')
    print('  🧹 Done')

=== Group 3: Principal Dependencies ===
  ✅ [201] Create principal
  ✅ [409] Create duplicate principal
  ✅ [404] Assign non-existent principal role
  ✅ [201] Assign valid principal role
  ✅ [500] Assign same principal role again
       💡 FINDING: 500 duplicate key — not idempotent
  ✅ [200] Self-rotate credentials
  ✅ [403] Root rotate other principal
       💡 FINDING: Admin cannot rotate others — by design
  ✅ [204] Delete principal with role assigned
       💡 FINDING: CASCADE — role assignment removed
  ✅ [200] Get principal role
       💡 FINDING: Principal Role not deleted
  ✅ [404] Get deleted principal

  🧹 Cleanup Group 3...
  🧹 Done


## Group 4 — Catalog Dependencies

In [298]:
print('=== Group 4: Catalog Dependencies ===')

def make_catalog(name, location):
    return {'catalog': {
            'name': name,
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': location,
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True
            }
        }}

safe_delete(f'{CAT}/dep-test-catalog/namespaces/dep-ns')
safe_delete(f'{BASE}/catalogs/dep-test-catalog')
safe_delete(f'{BASE}/catalogs/dep-test-catalog-s3')

try:
    # 1. Create catalog with s3a://
    r = requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json=make_catalog('dep-test-catalog', 's3a://data-catalog-bucket/dep-test/'))
    test('Catalog', 'Create catalog with s3a://', r, 500,
        '500 but resource created')

    # 2. Create duplicate catalog
    r = requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json=make_catalog('dep-test-catalog', 's3a://data-catalog-bucket/dep-test/'))
    test('Catalog', 'Create duplicate catalog', r, 409)

    # 3. Create catalog with s3:// scheme
    r = requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json=make_catalog('dep-test-catalog-s3', 's3://data-catalog-bucket/dep-test-s3/'))
    test('Catalog', 'Create catalog with s3:// scheme', r, 201,
         'FINDING: Polaris accepts s3:// — no scheme validation at creation')

    # 3-1. Delete catalog after emptying namespace
    # Expected: 204 (not 200 — Catalog API returns 204)
    requests.delete(f'{CAT}/dep-test-catalog/namespaces/dep-ns', headers=h(TOKEN_ROOT))
    r = requests.delete(f'{BASE}/catalogs/dep-test-catalog', headers=h(TOKEN_ROOT))
    test('Catalog', 'Delete catalog after emptying namespace', r, 204,
         'FINDING: 204 No Content after namespace removed')

    # Recreate for next test
    requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json=make_catalog('dep-test-catalog', 's3a://data-catalog-bucket/dep-test/'))

    # 3-2. Delete catalog WITH catalog role inside (no namespaces)
    requests.post(f'{BASE}/catalogs/dep-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'dep-test-role'}})

    r_roles = requests.get(f'{BASE}/catalogs/dep-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT))
    roles = [cr['name'] for cr in r_roles.json().get('roles', [])]
    print(f'  💡 Catalog roles before delete: {roles}')

    # Attempt to delete catalog WITH catalog role inside (no namespaces)
    # Expected: 400 — BLOCKED
    # FINDING: Polaris considers catalog roles as catalog content
    # Even catalog_admin (auto-created) blocks deletion
    r = requests.delete(f'{BASE}/catalogs/dep-test-catalog', headers=h(TOKEN_ROOT))
    test('Catalog', 'Delete catalog with catalog role inside', r, 400,
         'FINDING: BLOCKED — catalog roles count as content, must delete roles first')

    # Must delete catalog roles first before catalog can be deleted
    # catalog_admin is auto-created — must be deleted too
    for role_name in roles:
        if role_name != 'catalog_admin':  # try custom roles first
            requests.delete(
                f'{BASE}/catalogs/dep-test-catalog/catalog-roles/{role_name}',
                headers=h(TOKEN_ROOT))
    # Delete catalog_admin last
    requests.delete(
        f'{BASE}/catalogs/dep-test-catalog/catalog-roles/catalog_admin',
        headers=h(TOKEN_ROOT))

    # Verify all roles deleted
    r_roles = requests.get(f'{BASE}/catalogs/dep-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT))
    roles_after = [cr['name'] for cr in r_roles.json().get('roles', [])]
    print(f'  💡 Catalog roles after cleanup: {roles_after}')

    # Now delete catalog — all roles removed
    r = requests.delete(f'{BASE}/catalogs/dep-test-catalog', headers=h(TOKEN_ROOT))
    test('Catalog', 'Delete catalog after removing all catalog roles', r, 204,
         'FINDING: Succeeds only after ALL catalog roles (including catalog_admin) removed')

    # Recreate for remaining tests
    r = requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json=make_catalog('dep-test-catalog', 's3a://data-catalog-bucket/dep-test/'))
    test('Catalog', 'Recreate catalog for remaining tests', r, 201)

    # 4. Delete catalog — note: namespace creation returned 500 but may have been created
    # Check if namespace exists first
    ns_check = requests.get(f'{CAT}/dep-test-catalog/namespaces/dep-ns',
        headers=h(TOKEN_ROOT))
    if ns_check.status_code == 200:
        print('  💡 Namespace was created despite 500 response')
        requests.delete(f'{CAT}/dep-test-catalog/namespaces/dep-ns',
            headers=h(TOKEN_ROOT))

    r = requests.delete(f'{BASE}/catalogs/dep-test-catalog', headers=h(TOKEN_ROOT))
    test('Catalog', 'Delete catalog (empty after cleanup)', r, 204)

    # 5. Get deleted catalog
    r = requests.get(f'{BASE}/catalogs/dep-test-catalog', headers=h(TOKEN_ROOT))
    test('Catalog', 'Get deleted catalog', r, 404)

    # 6. Delete non-existent catalog
    r = requests.delete(f'{BASE}/catalogs/nonexistent-catalog', headers=h(TOKEN_ROOT))
    test('Catalog', 'Delete non-existent catalog', r, 404)

finally:
    print('\n  🧹 Cleanup Group 4...')
    safe_delete(f'{CAT}/dep-test-catalog/namespaces/dep-ns')
    safe_delete(f'{BASE}/catalogs/dep-test-catalog')
    safe_delete(f'{BASE}/catalogs/dep-test-catalog-s3')
    print('  🧹 Done')

=== Group 4: Catalog Dependencies ===
  ✅ [500] Create catalog with s3a://
       💡 500 but resource created
  ✅ [409] Create duplicate catalog
  ✅ [201] Create catalog with s3:// scheme
       💡 FINDING: Polaris accepts s3:// — no scheme validation at creation
  ✅ [204] Delete catalog after emptying namespace
       💡 FINDING: 204 No Content after namespace removed
  💡 Catalog roles before delete: ['catalog_admin', 'dep-test-role']
  ✅ [400] Delete catalog with catalog role inside
       💡 FINDING: BLOCKED — catalog roles count as content, must delete roles first
  💡 Catalog roles after cleanup: ['catalog_admin']
  ✅ [204] Delete catalog after removing all catalog roles
       💡 FINDING: Succeeds only after ALL catalog roles (including catalog_admin) removed
  ✅ [201] Recreate catalog for remaining tests
  ✅ [204] Delete catalog (empty after cleanup)
  ✅ [404] Get deleted catalog
  ✅ [404] Delete non-existent catalog

  🧹 Cleanup Group 4...
  🧹 Done


## Group 5 — Catalog Role Dependencies

In [299]:
print('=== Group 5: Catalog Role Dependencies ===')

safe_delete(f'{BASE}/principal-roles/cr-test-pr')
safe_delete(f'{BASE}/catalogs/cr-test-catalog')

try:
    r = requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json={'catalog': {
            'name': 'cr-test-catalog',
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://data-catalog-bucket/cr-test/',
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True
            }
        }})
    test('Catalog', 'Create catalog', r, 201)

    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'cr-test-pr'}})

    r = requests.post(f'{BASE}/catalogs/cr-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'cr-test-role'}})
    test('CatalogRole', 'Create catalog role', r, 201)

    r = requests.post(f'{BASE}/catalogs/cr-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'cr-test-role'}})
    test('CatalogRole', 'Create duplicate catalog role', r, 409)

    r = requests.put(
        f'{BASE}/catalogs/cr-test-catalog/catalog-roles/cr-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})
    test('CatalogRole', 'Grant privilege', r, 201)

    r = requests.put(
        f'{BASE}/catalogs/cr-test-catalog/catalog-roles/cr-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})
    test('CatalogRole', 'Grant duplicate privilege', r, 500,
         'FINDING: 500 duplicate key — not idempotent')

    r = requests.put(
        f'{BASE}/catalogs/cr-test-catalog/catalog-roles/nonexistent-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})
    test('CatalogRole', 'Grant to non-existent catalog role', r, 404)

    r = requests.put(
        f'{BASE}/principal-roles/cr-test-pr/catalog-roles/cr-test-catalog',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'cr-test-role'}})
    test('CatalogRole', 'Assign catalog role to principal role', r, 201)

    r = requests.delete(
        f'{BASE}/catalogs/cr-test-catalog/catalog-roles/cr-test-role',
        headers=h(TOKEN_ROOT))
    test('CatalogRole', 'Delete catalog role with grants + PR assignments', r, 204,
         'FINDING: CASCADE — grants and PR assignments deleted')

    r = requests.get(
        f'{BASE}/principal-roles/cr-test-pr/catalog-roles/cr-test-catalog',
        headers=h(TOKEN_ROOT))
    roles = r.json().get('roles', [])
    icon = '✅' if len(roles) == 0 else '⚠️ '
    print(f'  {icon} PR catalog roles after deletion: {roles}')

    r = requests.post(f'{BASE}/catalogs/nonexistent-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'test'}})
    test('CatalogRole', 'Create role on non-existent catalog', r, 404)

finally:
    print('\n  🧹 Cleanup Group 5...')
    safe_delete(f'{BASE}/catalogs/cr-test-catalog')
    safe_delete(f'{BASE}/principal-roles/cr-test-pr')
    print('  🧹 Done')

=== Group 5: Catalog Role Dependencies ===
  ✅ [201] Create catalog
  ✅ [201] Create catalog role
  ✅ [409] Create duplicate catalog role
  ✅ [201] Grant privilege
  ✅ [500] Grant duplicate privilege
       💡 FINDING: 500 duplicate key — not idempotent
  ✅ [404] Grant to non-existent catalog role
  ✅ [201] Assign catalog role to principal role
  ✅ [204] Delete catalog role with grants + PR assignments
       💡 FINDING: CASCADE — grants and PR assignments deleted
  ✅ PR catalog roles after deletion: []
  ✅ [404] Create role on non-existent catalog

  🧹 Cleanup Group 5...
  🧹 Done


## Group 6 — Namespace Dependencies

In [301]:
print('=== Group 6: Namespace Dependencies ===')

# ── Pre-cleanup ──────────────────────────────────────────────
safe_delete(f'{BASE}/principals/ns-test-svc')
safe_delete(f'{BASE}/principal-roles/ns-test-pr')
safe_delete(f'{BASE}/catalogs/ns-test-catalog')

SVC_TOKEN = ''

try:
    # ── Setup ────────────────────────────────────────────────
    # Catalog
    # Use real MinIO endpoint instead of dummy roleArn
    requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json={'catalog': {
            'name': 'ns-test-catalog',
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://data-catalog-bucket/cr-test/',
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True
            }
        }})
    
    # Catalog role — single grant only (no duplicate)
    requests.post(f'{BASE}/catalogs/ns-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT),
        json={'catalogRole': {'name': 'ns-admin-role'}})
    requests.put(f'{BASE}/catalogs/ns-test-catalog/catalog-roles/ns-admin-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}})

    # Principal role
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'ns-test-pr'}})
    requests.put(f'{BASE}/principal-roles/ns-test-pr/catalog-roles/ns-test-catalog',
        headers=h(TOKEN_ROOT),
        json={'catalogRole': {'name': 'ns-admin-role'}})

    # Principal — save credentials immediately
    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'ns-test-svc', 'type': 'SERVICE'})
    creds = r.json().get('credentials', {})
    requests.put(f'{BASE}/principals/ns-test-svc/principal-roles',
        headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'ns-test-pr'}})

    # Get service token
    SVC_TOKEN = get_token(
        creds.get('clientId', ''), creds.get('clientSecret', '')
    ).json().get('access_token', '')
    print(f'  Setup: {"✅" if SVC_TOKEN else "❌"} service token obtained')

    # ── Tests ────────────────────────────────────────────────

    # 1. Create top-level namespace
    r = requests.post(f'{CAT}/ns-test-catalog/namespaces',
        headers=h(SVC_TOKEN),
        json={'namespace': ['parent'], 'properties': {}})
    test('Namespace', 'Create top-level namespace', r, 500,
        '500 but resource created')

    # 2. Create nested namespace
    r = requests.post(f'{CAT}/ns-test-catalog/namespaces',
        headers=h(SVC_TOKEN),
        json={'namespace': ['parent', 'child'], 'properties': {}})
    test('Namespace', 'Create nested namespace (parent.child)', r, 500,
        '500 but resource created')

    # 3. Create deeply nested namespace (3 levels)
    r = requests.post(f'{CAT}/ns-test-catalog/namespaces',
        headers=h(SVC_TOKEN),
        json={'namespace': ['parent', 'child', 'grandchild'], 'properties': {}})
    test('Namespace', 'Create deeply nested namespace (3 levels)', r, 500,
        '500 but resource created')

    # 4. Create duplicate namespace
    r = requests.post(f'{CAT}/ns-test-catalog/namespaces',
        headers=h(SVC_TOKEN),
        json={'namespace': ['parent'], 'properties': {}})
    test('Namespace', 'Create duplicate namespace', r, 409)

    # 5. Create child without parent existing
    r = requests.post(f'{CAT}/ns-test-catalog/namespaces',
        headers=h(SVC_TOKEN),
        json={'namespace': ['nonexistent-parent', 'child'], 'properties': {}})
    test('Namespace', 'Create child without parent existing', r, 404,
         'FINDING: BLOCKED — parent must exist first')

    # 6. Create table inside namespace
    r = requests.post(f'{CAT}/ns-test-catalog/namespaces/parent/tables',
        headers=h(SVC_TOKEN),
        json={'name': 'dep-test-table',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'partition-spec': {'spec-id': 0, 'fields': []},
              'write-order': {'order-id': 0, 'fields': []},
              'stage-create': False, 'properties': {}})
    test('Namespace', 'Create table inside namespace', r, 200)

    # 7. Delete namespace WITH table inside
    r = requests.delete(f'{CAT}/ns-test-catalog/namespaces/parent',
        headers=h(SVC_TOKEN))
    test('Namespace', 'Delete namespace with table inside', r, 409,
         'FINDING: BLOCKED — namespace must be empty first')

    # 8. Delete namespace WITH child namespace
    r = requests.delete(f'{CAT}/ns-test-catalog/namespaces/parent%1Fchild',
        headers=h(SVC_TOKEN))
    test('Namespace', 'Delete namespace with child namespace', r, 409,
         'FINDING: BLOCKED — must delete children first')

    # 9. Delete leaf namespace (empty)
    r = requests.delete(
        f'{CAT}/ns-test-catalog/namespaces/parent%1Fchild%1Fgrandchild',
        headers=h(SVC_TOKEN))
    test('Namespace', 'Delete leaf namespace (empty)', r, 204)

finally:
    print('\n  🧹 Cleanup Group 6...')
    t = SVC_TOKEN or TOKEN_ROOT
    safe_delete(f'{CAT}/ns-test-catalog/namespaces/parent/tables/dep-test-table', t)
    safe_delete(f'{CAT}/ns-test-catalog/namespaces/parent%1Fchild%1Fgrandchild', t)
    safe_delete(f'{CAT}/ns-test-catalog/namespaces/parent%1Fchild', t)
    safe_delete(f'{CAT}/ns-test-catalog/namespaces/parent', t)
    safe_delete(f'{BASE}/principals/ns-test-svc')
    safe_delete(f'{BASE}/principal-roles/ns-test-pr')
    safe_delete(f'{BASE}/catalogs/ns-test-catalog')
    print('  🧹 Done')

=== Group 6: Namespace Dependencies ===
  Setup: ✅ service token obtained
  ✅ [500] Create top-level namespace
       💡 500 but resource created
  ✅ [500] Create nested namespace (parent.child)
       💡 500 but resource created
  ✅ [500] Create deeply nested namespace (3 levels)
       💡 500 but resource created
  ✅ [409] Create duplicate namespace
  ✅ [404] Create child without parent existing
       💡 FINDING: BLOCKED — parent must exist first
  ✅ [200] Create table inside namespace
  ✅ [409] Delete namespace with table inside
       💡 FINDING: BLOCKED — namespace must be empty first
  ✅ [409] Delete namespace with child namespace
       💡 FINDING: BLOCKED — must delete children first
  ✅ [204] Delete leaf namespace (empty)

  🧹 Cleanup Group 6...
  🧹 Done


## Group 7 — Table Dependencies

In [303]:
print('=== Group 7: Table Dependencies ===')

safe_delete(f'{BASE}/principals/tbl-test-svc')
safe_delete(f'{BASE}/principal-roles/tbl-test-pr')
safe_delete(f'{BASE}/catalogs/tbl-test-catalog')

SVC_TOKEN2 = ''

def make_table(name):
    return {'name': name,
            'schema': {'type': 'struct', 'schema-id': 0,
                'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True},
                           {'id': 2, 'name': 'value', 'type': 'string', 'required': False}]},
            'partition-spec': {'spec-id': 0, 'fields': []},
            'write-order': {'order-id': 0, 'fields': []},
            'stage-create': False, 'properties': {}}

try:
    requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json={'catalog': {
            'name': 'tbl-test-catalog',
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://data-catalog-bucket/cr-test/',
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True
            }
        }})
    requests.post(f'{BASE}/catalogs/tbl-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'tbl-admin-role'}})
    requests.put(f'{BASE}/catalogs/tbl-test-catalog/catalog-roles/tbl-admin-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}})
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'tbl-test-pr'}})
    requests.put(f'{BASE}/principal-roles/tbl-test-pr/catalog-roles/tbl-test-catalog',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'tbl-admin-role'}})
    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'tbl-test-svc', 'type': 'SERVICE'})
    creds = r.json().get('credentials', {})
    requests.put(f'{BASE}/principals/tbl-test-svc/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'tbl-test-pr'}})
    SVC_TOKEN2 = get_token(
        creds.get('clientId', ''), creds.get('clientSecret', '')
    ).json().get('access_token', '')
    requests.post(f'{CAT}/tbl-test-catalog/namespaces', headers=h(SVC_TOKEN2),
        json={'namespace': ['tbl-ns'], 'properties': {}})
    print(f'  Service token: {"✅" if SVC_TOKEN2 else "❌"}')

    r = requests.post(f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/tables',
        headers=h(SVC_TOKEN2), json=make_table('test-table'))
    test('Table', 'Create table', r, 200)

    r = requests.post(f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/tables',
        headers=h(SVC_TOKEN2), json=make_table('test-table'))
    test('Table', 'Create duplicate table', r, 409)

    r = requests.post(f'{CAT}/tbl-test-catalog/namespaces/nonexistent-ns/tables',
        headers=h(SVC_TOKEN2), json=make_table('orphan-table'))
    test('Table', 'Create table in non-existent namespace', r, 404)

    r = requests.post(f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/views',
        headers=h(SVC_TOKEN2),
        json={'name': 'invalid-view',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'view-version': {'version-id': 1, 'schema-id': 0, 'timestamp-ms': 0,
                  'summary': {'engine-name': 'spark'},
                  'default-namespace': ['tbl-ns'],
                  'representations': [{'type': 'sql',
                      'sql': 'SELECT * FROM completely_nonexistent_table_xyz',
                      'dialect': 'spark'}]},
              'properties': {}})
    test('Table', 'Create view with invalid SQL', r, 500,
         'FINDING: No SQL validation — stores definition only. 500 but resource created')

    # Verify view was actually created and SQL definition stored as-is
    r_check = requests.get(
        f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/views/invalid-view',
        headers=h(SVC_TOKEN2))
    if r_check.status_code == 200:
        body = r_check.json()
        # Extract stored SQL from view representations
        representations = (body.get('metadata', {})
                               .get('versions', [{}])[0]
                               .get('representations', [{}]))
        stored_sql = representations[0].get('sql', '') if representations else ''
        print(f'  ✅ View exists in Polaris — SQL stored as-is:')
        print(f'     → {stored_sql}')
        print(f'     FINDING: Polaris stores invalid SQL without validation')
    else:
        print(f'  ❌ View NOT found after creation returned 200 — status: {r_check.status_code}')

    # Also verify via list views
    r_list = requests.get(
        f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/views',
        headers=h(SVC_TOKEN2))
    views = [v.get('name') for v in r_list.json().get('identifiers', [])]
    print(f'  💡 Views in tbl-ns: {views}')

    requests.post(f'{BASE}/catalogs/tbl-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'tbl-reader-role'}})
    requests.put(f'{BASE}/catalogs/tbl-test-catalog/catalog-roles/tbl-reader-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'table', 'namespace': ['tbl-ns'],
                        'tableName': 'test-table', 'privilege': 'TABLE_READ_DATA'}})

    r = requests.delete(
        f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/tables/test-table',
        headers=h(SVC_TOKEN2))
    test('Table', 'Delete table with scoped grant pointing to it', r, 204,
         'Does Polaris block or leave orphaned grant?')

    r = requests.get(
        f'{BASE}/catalogs/tbl-test-catalog/catalog-roles/tbl-reader-role/grants',
        headers=h(TOKEN_ROOT))
    grants = r.json().get('grants', [])
    icon = '⚠️ ' if grants else '✅'
    print(f'  {icon} Grants after table deletion: {grants}')
    print(f'       FINDING: Orphaned grants {"REMAIN" if grants else "cleaned up"}')

    r = requests.delete(
    f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/views/invalid-view',
    headers=h(SVC_TOKEN2))
    test('Table', 'Delete view', r, 204)

finally:
    print('\n  🧹 Cleanup Group 7...')
    t = SVC_TOKEN2 or TOKEN_ROOT
    safe_delete(f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/views/invalid-view', t)
    safe_delete(f'{CAT}/tbl-test-catalog/namespaces/tbl-ns/tables/test-table', t)
    safe_delete(f'{CAT}/tbl-test-catalog/namespaces/tbl-ns', t)
    safe_delete(f'{BASE}/principals/tbl-test-svc')
    safe_delete(f'{BASE}/principal-roles/tbl-test-pr')
    safe_delete(f'{BASE}/catalogs/tbl-test-catalog')
    print('  🧹 Done')

=== Group 7: Table Dependencies ===
  Service token: ✅
  ✅ [200] Create table
  ✅ [409] Create duplicate table
  ✅ [404] Create table in non-existent namespace
  ✅ [500] Create view with invalid SQL
       💡 FINDING: No SQL validation — stores definition only. 500 but resource created
  ✅ View exists in Polaris — SQL stored as-is:
     → SELECT * FROM completely_nonexistent_table_xyz
     FINDING: Polaris stores invalid SQL without validation
  💡 Views in tbl-ns: ['invalid-view']
  ✅ [204] Delete table with scoped grant pointing to it
       💡 Does Polaris block or leave orphaned grant?
  ✅ Grants after table deletion: []
       FINDING: Orphaned grants cleaned up
  ✅ [204] Delete view

  🧹 Cleanup Group 7...
  🧹 Done


## Group 8 — Full Lifecycle Sequential Flow

In [305]:
print('=== Group 8: Full Lifecycle Sequential Flow ===')

MINIO_ENDPOINT          = MINIO_ENDPOINT  # from init_env (src/config)
MINIO_ENDPOINT_INTERNAL = 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000'


# ── Pre-cleanup (get service token first) ────────────────────
print('  🧹 Pre-cleanup...')

# Get existing lifecycle-svc credentials via rotate to get a usable token
_cleanup_token = TOKEN_ROOT  # fallback

# Try to get service token for catalog cleanup
_pr = requests.get(f'{BASE}/principals/lifecycle-svc', headers=h(TOKEN_ROOT))
if _pr.status_code == 200:
    # Principal exists — rotate to get credentials
    _rot = requests.post(
        f'{BASE}/principals/lifecycle-svc/rotate',
        headers=h(TOKEN_ROOT))
    # Root can't rotate others — use delete+recreate approach
    # Instead just delete via Management API (catalog delete cascades if empty)
    pass

# Delete in correct order using root for management, root for catalog too
# root CAN use Catalog API (returns 500 on some ops but silently ignored by safe_delete)
safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/views/lifecycle-view')
safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/tables/lifecycle-table')
safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested')
safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns')
safe_delete(f'{BASE}/catalogs/lifecycle-catalog/catalog-roles/lifecycle-role')
safe_delete(f'{BASE}/catalogs/lifecycle-catalog')
safe_delete(f'{BASE}/principals/lifecycle-svc')
safe_delete(f'{BASE}/principal-roles/lifecycle-pr')
print('  🧹 Pre-cleanup done')

LC_TOKEN = ''
lc_creds = {}

try:
    print('── CREATE PHASE ──')

    r = requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'lifecycle-pr'}})
    test('Lifecycle', 'Create principal role', r, 201)

    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'lifecycle-svc', 'type': 'SERVICE'})
    test('Lifecycle', 'Create principal', r, 201)
    lc_creds = r.json().get('credentials', {})

    r = requests.put(f'{BASE}/principals/lifecycle-svc/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'lifecycle-pr'}})
    test('Lifecycle', 'Assign principal role', r, 201)

    r = requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json={'catalog': {
            'name': 'lifecycle-catalog',
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://data-catalog-bucket/lifecycle-catalog/',
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True,
                'endpoint': MINIO_ENDPOINT,
                # endpointInternal: Polaris server uses this to reach MinIO inside K8s
                # Without this, Polaris tries to resolve MinIO via AWS SDK → NullPointerException
                'endpointInternal': MINIO_ENDPOINT_INTERNAL
                # no roleArn — MinIO does not use AWS IAM
            }
        }})
    test('Lifecycle', 'Create catalog', r, 201)

    r = requests.post(f'{BASE}/catalogs/lifecycle-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'lifecycle-role'}})
    test('Lifecycle', 'Create catalog role', r, 201)

    r = requests.put(
        f'{BASE}/catalogs/lifecycle-catalog/catalog-roles/lifecycle-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}})
    test('Lifecycle', 'Grant privilege', r, 201)

    r = requests.put(
        f'{BASE}/principal-roles/lifecycle-pr/catalog-roles/lifecycle-catalog',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'lifecycle-role'}})
    test('Lifecycle', 'Assign catalog role to principal role', r, 201)

    LC_TOKEN = get_token(
        lc_creds.get('clientId', ''), lc_creds.get('clientSecret', '')
    ).json().get('access_token', '')
    print(f'  ✅ Service token obtained')

    r = requests.post(f'{CAT}/lifecycle-catalog/namespaces', headers=h(LC_TOKEN),
        json={'namespace': ['lc-ns'], 'properties': {}})
    test('Lifecycle', 'Create namespace', r, 500,
        '500 but resource created')

    r = requests.post(f'{CAT}/lifecycle-catalog/namespaces', headers=h(LC_TOKEN),
        json={'namespace': ['lc-ns', 'nested'], 'properties': {}})
    test('Lifecycle', 'Create nested namespace', r, 500,
        '500 but resource created')

    r = requests.post(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/tables',
        headers=h(LC_TOKEN),
        json={'name': 'lifecycle-table',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'partition-spec': {'spec-id': 0, 'fields': []},
              'write-order': {'order-id': 0, 'fields': []},
              'stage-create': False, 'properties': {}})
    test('Lifecycle', 'Create table', r, 200)

    r = requests.post(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/views',
        headers=h(LC_TOKEN),
        json={'name': 'lifecycle-view',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'view-version': {'version-id': 1, 'schema-id': 0, 'timestamp-ms': 0,
                  'summary': {'engine-name': 'spark'},
                  'default-namespace': ['lc-ns', 'nested'],
                  'representations': [{'type': 'sql',
                      'sql': 'SELECT id FROM lifecycle_table', 'dialect': 'spark'}]},
              'properties': {}})
    test('Lifecycle', 'Create view', r, 500,
        '500 but resource created')

    print('\n── USE PHASE ──')

    r = requests.get(f'{CAT}/lifecycle-catalog/namespaces', headers=h(LC_TOKEN))
    test('Lifecycle', 'List namespaces', r, 200)

    r = requests.get(
        f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/tables', headers=h(LC_TOKEN))
    test('Lifecycle', 'List tables', r, 200)

    r = requests.get(
        f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/tables/lifecycle-table',
        headers=h(LC_TOKEN))
    test('Lifecycle', 'Load table metadata', r, 200)

    print('\n── DELETE PHASE (reverse order) ──')

    r = requests.delete(
        f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/views/lifecycle-view',
        headers=h(LC_TOKEN))
    test('Lifecycle', 'Delete view', r, 204)

    r = requests.delete(
        f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/tables/lifecycle-table',
        headers=h(LC_TOKEN))
    test('Lifecycle', 'Delete table', r, 204)

    r = requests.delete(
        f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested', headers=h(LC_TOKEN))
    test('Lifecycle', 'Delete nested namespace (empty)', r, 204)

    r = requests.delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns', headers=h(LC_TOKEN))
    test('Lifecycle', 'Delete parent namespace (empty)', r, 204)

    r = requests.delete(
        f'{BASE}/catalogs/lifecycle-catalog/catalog-roles/lifecycle-role',
        headers=h(TOKEN_ROOT))
    test('Lifecycle', 'Delete catalog role', r, 204)

    r = requests.delete(f'{BASE}/catalogs/lifecycle-catalog', headers=h(TOKEN_ROOT))
    test('Lifecycle', 'Delete catalog', r, 204)

    r = requests.delete(f'{BASE}/principals/lifecycle-svc', headers=h(TOKEN_ROOT))
    test('Lifecycle', 'Delete principal', r, 204)

    r = requests.delete(f'{BASE}/principal-roles/lifecycle-pr', headers=h(TOKEN_ROOT))
    test('Lifecycle', 'Delete principal role', r, 204)

    print('\n✅ Full lifecycle complete')

finally:
    print('\n  🧹 Cleanup Group 8...')
    t = LC_TOKEN or TOKEN_ROOT
    safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/views/lifecycle-view', t)
    safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested/tables/lifecycle-table', t)
    safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns%1Fnested', t)
    safe_delete(f'{CAT}/lifecycle-catalog/namespaces/lc-ns', t)
    safe_delete(f'{BASE}/catalogs/lifecycle-catalog/catalog-roles/lifecycle-role')
    safe_delete(f'{BASE}/catalogs/lifecycle-catalog')
    safe_delete(f'{BASE}/principals/lifecycle-svc')
    safe_delete(f'{BASE}/principal-roles/lifecycle-pr')
    print('  🧹 Done')

=== Group 8: Full Lifecycle Sequential Flow ===
  🧹 Pre-cleanup...
  🧹 Pre-cleanup done
── CREATE PHASE ──
  ✅ [201] Create principal role
  ✅ [201] Create principal
  ✅ [201] Assign principal role
  ✅ [201] Create catalog
  ✅ [201] Create catalog role
  ✅ [201] Grant privilege
  ✅ [201] Assign catalog role to principal role
  ✅ Service token obtained
  ✅ [500] Create namespace
       💡 500 but resource created
  ✅ [500] Create nested namespace
       💡 500 but resource created
  ✅ [200] Create table
  ✅ [500] Create view
       💡 500 but resource created

── USE PHASE ──
  ✅ [200] List namespaces
  ✅ [200] List tables
  ✅ [200] Load table metadata

── DELETE PHASE (reverse order) ──
  ✅ [204] Delete view
  ✅ [204] Delete table
  ✅ [204] Delete nested namespace (empty)
  ✅ [204] Delete parent namespace (empty)
  ✅ [204] Delete catalog role
  ✅ [204] Delete catalog
  ✅ [204] Delete principal
  ✅ [204] Delete principal role

✅ Full lifecycle complete

  🧹 Cleanup Group 8...
  🧹 Done


## Group 9 — Error Response Format

In [306]:
print('=== Group 9: Error Response Format ===')
print('Verify consistent error structure across all HTTP status codes\n')

error_cases = [
    ('400 Bad Request',
     requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
         json={'catalog': {'name': '', 'type': 'INTERNAL'}})),
    ('401 Unauthorized',
     requests.get(f'{BASE}/principals',
         headers={'Authorization': 'Bearer INVALID', 'Polaris-Realm': REALM})),
    ('404 Not Found',
     requests.get(f'{BASE}/catalogs/nonexistent-xyz', headers=h(TOKEN_ROOT))),
    ('409 Conflict',
     requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
         json={'principalRole': {'name': 'service_admin'}})),
    ('500 Internal Error',
     requests.put(f'{BASE}/principal-roles/service_admin/catalog-roles/test-catalog',
         headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'catalog_admin'}})),
]

print(f'{"Status":<22} {"HTTP":<6} {"error.code":<12} {"Type":<35} Message')
print('-' * 120)
for label, r in error_cases:
    try:
        body  = r.json()
        err   = body.get('error', body)
        etype = err.get('type', 'N/A')[:35]
        emsg  = err.get('message', str(body))[:50]
        ecode = err.get('code', '?')
        print(f'{label:<22} {r.status_code:<6} {str(ecode):<12} {etype:<35} {emsg}')
    except:
        print(f'{label:<22} {r.status_code:<6} Non-JSON: {r.text[:60]}')

print('\n💡 Standard error structure:')
print(json.dumps({'error': {'message': 'Human readable message',
                             'type': 'ExceptionClassName',
                             'code': 404}}, indent=2))

=== Group 9: Error Response Format ===
Verify consistent error structure across all HTTP status codes

Status                 HTTP   error.code   Type                                Message
------------------------------------------------------------------------------------------------------------------------
400 Bad Request        400    400          ResteasyReactiveViolationException  Invalid value: createCatalog.arg0.catalog.properti
401 Unauthorized       401    Non-JSON: 
404 Not Found          404    404          NotFoundException                   TopLevelEntity of type CATALOG does not exist: non
409 Conflict           409    409          AlreadyExistsException              Cannot create PrincipalRole service_admin. Princip
500 Internal Error     404    404          NotFoundException                   Entity test-catalog not found when trying to assig

💡 Standard error structure:
{
  "error": {
    "message": "Human readable message",
    "type": "ExceptionClassName",
    "code

## Group 10 — Additional Edge Cases — Auth & Principal

In [307]:
print('=== Group 10: Additional Edge Cases — Auth & Principal ===')

# Pre-cleanup
safe_delete(f'{BASE}/principals/edge-test-user')
safe_delete(f'{BASE}/principal-roles/edge-test-pr')

try:
    # Setup
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'edge-test-pr'}})
    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'edge-test-user', 'type': 'SERVICE'})
    creds = r.json().get('credentials', {})
    EDGE_CID     = creds.get('clientId', '')
    EDGE_CSECRET = creds.get('clientSecret', '')

    # Assign principal role first
    requests.put(f'{BASE}/principals/edge-test-user/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'edge-test-pr'}})

    # Test 1: Get token for new principal — should work immediately
    r = get_token(EDGE_CID, EDGE_CSECRET)
    test('Auth+', 'Get token for new principal', r, 200)
    EDGE_TOKEN = r.json().get('access_token', '')

    # Test 2: Token works before deletion
    r = get_token(EDGE_CID, EDGE_CSECRET)
    test('Auth+', 'Token valid before principal deleted', r, 200,
         'FINDING: Token works — verified by re-issuing token with same credentials')

    # Test 3: Delete principal
    requests.delete(f'{BASE}/principals/edge-test-user', headers=h(TOKEN_ROOT))

    # Test 4: Use old token after principal deleted
    # Key question: does Polaris invalidate tokens when principal is deleted?
    # JWT is stateless — Polaris may still accept it until expiry
    r = get_token(EDGE_CID, EDGE_CSECRET)
    test('Auth+', 'Get new token after principal deleted', r, 401,
         'FINDING: Cannot get new token — principal deleted from principal_authentication_data')

    # Test 5: Use previously issued token after principal deleted
    r = requests.get(f'{BASE}/principals', headers=h(EDGE_TOKEN))
    test('Auth+', 'Old token still works after principal deleted', r, 401,
         'FINDING: Does Polaris invalidate existing JWT on principal delete?')
    
    # Test 5: Assign principal role to non-existent principal
    r = requests.put(f'{BASE}/principals/nonexistent-principal/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'edge-test-pr'}})
    test('Auth+', 'Assign role to non-existent principal', r, 404)

    # Test 6: Delete non-existent principal
    r = requests.delete(f'{BASE}/principals/nonexistent-xyz', headers=h(TOKEN_ROOT))
    test('Auth+', 'Delete non-existent principal', r, 404)

    # Test 7: List all principals
    r = requests.get(f'{BASE}/principals', headers=h(TOKEN_ROOT))
    test('Auth+', 'List all principals', r, 200)
    names = [p['name'] for p in r.json().get('principals', [])]
    print(f'  💡 Principals: {names}')

    # Test 8: Service token calling Management API list principals
    # Service principal with no SERVICE_MANAGE_ACCESS — can it list principals?
    r2 = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'edge-test-user', 'type': 'SERVICE'})
    creds2 = r2.json().get('credentials', {})
    svc_tok = get_token(creds2.get('clientId',''), creds2.get('clientSecret','')).json().get('access_token','')
    r = requests.get(f'{BASE}/principals', headers=h(svc_tok))
    test('Auth+', 'Service token calling Management API list principals', r, 403,
         'FINDING: Service principal cannot call Management API without SERVICE_MANAGE_ACCESS')

finally:
    print('\n  🧹 Cleanup Group 10...')
    safe_delete(f'{BASE}/principals/edge-test-user')
    safe_delete(f'{BASE}/principal-roles/edge-test-pr')
    print('  🧹 Done')

=== Group 10: Additional Edge Cases — Auth & Principal ===
  ✅ [200] Get token for new principal
  ✅ [200] Token valid before principal deleted
       💡 FINDING: Token works — verified by re-issuing token with same credentials
  ✅ [401] Get new token after principal deleted
       💡 FINDING: Cannot get new token — principal deleted from principal_authentication_data
  ✅ [401] Old token still works after principal deleted
       💡 FINDING: Does Polaris invalidate existing JWT on principal delete?
  ✅ [404] Assign role to non-existent principal
  ✅ [404] Delete non-existent principal
  ✅ [200] List all principals
  💡 Principals: ['root']
  ✅ [403] Service token calling Management API list principals
       💡 FINDING: Service principal cannot call Management API without SERVICE_MANAGE_ACCESS

  🧹 Cleanup Group 10...
  🧹 Done


## Group 11 - Additional Edge Cases — Catalog Role & Grant

In [308]:
print('=== Group 11: Additional Edge Cases — Catalog Role & Grant ===')

safe_delete(f'{BASE}/principal-roles/grant-test-pr')
safe_delete(f'{BASE}/catalogs/grant-test-catalog')

try:
    # Setup
    requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json={'catalog': {
            'name': 'grant-test-catalog', 'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://data-catalog-bucket/grant-test/',
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True,
                'endpoint': MINIO_ENDPOINT,
                'endpointInternal': 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000'
            }
        }})
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'grant-test-pr'}})
    requests.post(f'{BASE}/catalogs/grant-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'grant-test-role'}})
    requests.put(f'{BASE}/catalogs/grant-test-catalog/catalog-roles/grant-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})

    # Test 1: Assign same catalog role to principal role twice
    # Parallel to duplicate grant — is assignment idempotent?
    requests.put(f'{BASE}/principal-roles/grant-test-pr/catalog-roles/grant-test-catalog',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'grant-test-role'}})
    r = requests.put(f'{BASE}/principal-roles/grant-test-pr/catalog-roles/grant-test-catalog',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'grant-test-role'}})
    test('CatalogRole+', 'Assign same catalog role to PR twice', r, 500,
         'FINDING: 500 duplicate key — catalog role assignment NOT idempotent')

    # Test 2: Revoke grant via REST API DELETE → 405 Method Not Allowed
    # FINDING: DELETE /grants not implemented in 1.3.0 REST API
    # Grant revocation via REST is NOT possible — CLI only
    r = requests.delete(
        f'{BASE}/catalogs/grant-test-catalog/catalog-roles/grant-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})
    test('CatalogRole+', 'Revoke grant via REST API (DELETE /grants)', r, 405,
         'FINDING: 405 — DELETE /grants NOT implemented in 1.3.0 REST API. '
         'Grant revocation only available via Polaris CLI')

    # Test 3: Revoke non-existent grant via DELETE → also 405
    r = requests.delete(
        f'{BASE}/catalogs/grant-test-catalog/catalog-roles/grant-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}})
    test('CatalogRole+', 'Revoke non-existent grant via REST API', r, 405,
         'FINDING: 405 regardless — method not implemented')

    # Test 4: Re-grant same privilege → 500 (grant still exists, cannot revoke via REST)
    r = requests.put(
        f'{BASE}/catalogs/grant-test-catalog/catalog-roles/grant-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})
    test('CatalogRole+', 'Re-grant existing privilege', r, 500,
         'FINDING: 500 — grant exists, cannot revoke via REST. '
         'Only workaround: delete catalog role + recreate')

    # Test 5: Workaround — delete catalog role and recreate to modify grants
    # This is the ONLY way to change grants via REST API in Polaris 1.3.0
    requests.delete(
        f'{BASE}/catalogs/grant-test-catalog/catalog-roles/grant-test-role',
        headers=h(TOKEN_ROOT))
    requests.post(f'{BASE}/catalogs/grant-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'grant-test-role'}})
    r = requests.put(
        f'{BASE}/catalogs/grant-test-catalog/catalog-roles/grant-test-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}})
    test('CatalogRole+', 'Grant after delete+recreate catalog role', r, 201,
         'FINDING: Only REST workaround — delete catalog role and recreate with desired grants')

finally:
    print('\n  🧹 Cleanup Group 11...')
    safe_delete(f'{BASE}/catalogs/grant-test-catalog')
    safe_delete(f'{BASE}/principal-roles/grant-test-pr')
    print('  🧹 Done')

=== Group 11: Additional Edge Cases — Catalog Role & Grant ===
  ✅ [500] Assign same catalog role to PR twice
       💡 FINDING: 500 duplicate key — catalog role assignment NOT idempotent
  ✅ [405] Revoke grant via REST API (DELETE /grants)
       💡 FINDING: 405 — DELETE /grants NOT implemented in 1.3.0 REST API. Grant revocation only available via Polaris CLI
  ✅ [405] Revoke non-existent grant via REST API
       💡 FINDING: 405 regardless — method not implemented
  ✅ [500] Re-grant existing privilege
       💡 FINDING: 500 — grant exists, cannot revoke via REST. Only workaround: delete catalog role + recreate
  ✅ [201] Grant after delete+recreate catalog role
       💡 FINDING: Only REST workaround — delete catalog role and recreate with desired grants

  🧹 Cleanup Group 11...
  🧹 Done


## Group 12 - Additional Edge Cases — Namespace & View

In [309]:
print('=== Group 12: Additional Edge Cases — Namespace & View ==='),

safe_delete(f'{BASE}/principals/ns2-test-svc')
safe_delete(f'{BASE}/principal-roles/ns2-test-pr')
safe_delete(f'{BASE}/catalogs/ns2-test-catalog')

SVC_TOKEN3 = ''

try:
    # Setup
    requests.post(f'{BASE}/catalogs', headers=h(TOKEN_ROOT),
        json={'catalog': {
            'name': 'ns2-test-catalog', 'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://data-catalog-bucket/ns2-test/',
                'polaris.config.drop-with-purge.enabled': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'pathStyleAccess': True,
                'endpoint': MINIO_ENDPOINT,
                'endpointInternal': 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000'
            }
        }})
    requests.post(f'{BASE}/catalogs/ns2-test-catalog/catalog-roles',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'ns2-admin-role'}})
    requests.put(f'{BASE}/catalogs/ns2-test-catalog/catalog-roles/ns2-admin-role/grants',
        headers=h(TOKEN_ROOT),
        json={'grant': {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}})
    requests.post(f'{BASE}/principal-roles', headers=h(TOKEN_ROOT),
        json={'principalRole': {'name': 'ns2-test-pr'}})
    requests.put(f'{BASE}/principal-roles/ns2-test-pr/catalog-roles/ns2-test-catalog',
        headers=h(TOKEN_ROOT), json={'catalogRole': {'name': 'ns2-admin-role'}})
    r = requests.post(f'{BASE}/principals', headers=h(TOKEN_ROOT),
        json={'name': 'ns2-test-svc', 'type': 'SERVICE'})
    creds = r.json().get('credentials', {})
    requests.put(f'{BASE}/principals/ns2-test-svc/principal-roles',
        headers=h(TOKEN_ROOT), json={'principalRole': {'name': 'ns2-test-pr'}})
    SVC_TOKEN3 = get_token(
        creds.get('clientId', ''), creds.get('clientSecret', '')
    ).json().get('access_token', '')

    # Create namespace for tests
    requests.post(f'{CAT}/ns2-test-catalog/namespaces', headers=h(SVC_TOKEN3),
        json={'namespace': ['test-ns'], 'properties': {}})
    print(f'  Setup: {"✅" if SVC_TOKEN3 else "❌"} service token obtained')

    # Test 1: Get namespace properties
    r = requests.get(f'{CAT}/ns2-test-catalog/namespaces/test-ns',
        headers=h(SVC_TOKEN3))
    test('Namespace+', 'Get namespace properties', r, 200)
    if r.status_code == 200:
        props = r.json().get('properties', {})
        print(f'  💡 Namespace properties: {props}')

    # Test 2: Update namespace properties
    r = requests.post(f'{CAT}/ns2-test-catalog/namespaces/test-ns/properties',
        headers=h(SVC_TOKEN3),
        json={'updates': {'owner': 'data-team', 'description': 'test namespace'}})
    test('Namespace+', 'Update namespace properties', r, 200,
         'POST /namespaces/{ns}/properties with updates key')

    # Test 3: List namespaces with parent filter
    # Create nested namespace first
    requests.post(f'{CAT}/ns2-test-catalog/namespaces', headers=h(SVC_TOKEN3),
        json={'namespace': ['test-ns', 'child1'], 'properties': {}})
    requests.post(f'{CAT}/ns2-test-catalog/namespaces', headers=h(SVC_TOKEN3),
        json={'namespace': ['test-ns', 'child2'], 'properties': {}})
    # List only children of test-ns using parent filter
    r = requests.get(f'{CAT}/ns2-test-catalog/namespaces',
        headers=h(SVC_TOKEN3),
        params={'parent': 'test-ns'})
    test('Namespace+', 'List namespaces with parent filter', r, 200,
         'GET /namespaces?parent=test-ns returns only direct children')
    if r.status_code == 200:
        children = r.json().get('namespaces', [])
        print(f'  💡 Children of test-ns: {children}')

    # Test 4: Create view in non-existent namespace
    r = requests.post(f'{CAT}/ns2-test-catalog/namespaces/nonexistent-ns/views',
        headers=h(SVC_TOKEN3),
        json={'name': 'test-view',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'view-version': {'version-id': 1, 'schema-id': 0, 'timestamp-ms': 0,
                  'summary': {'engine-name': 'spark'},
                  'default-namespace': ['nonexistent-ns'],
                  'representations': [{'type': 'sql',
                      'sql': 'SELECT 1 AS id', 'dialect': 'spark'}]},
              'properties': {}})
    test('Namespace+', 'Create view in non-existent namespace', r, 404,
         'FINDING: Parallel to table in non-existent namespace')

    # Test 5: Create duplicate view
    requests.post(f'{CAT}/ns2-test-catalog/namespaces/test-ns/views',
        headers=h(SVC_TOKEN3),
        json={'name': 'dup-view',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'view-version': {'version-id': 1, 'schema-id': 0, 'timestamp-ms': 0,
                  'summary': {'engine-name': 'spark'},
                  'default-namespace': ['test-ns'],
                  'representations': [{'type': 'sql',
                      'sql': 'SELECT 1 AS id', 'dialect': 'spark'}]},
              'properties': {}})
    r = requests.post(f'{CAT}/ns2-test-catalog/namespaces/test-ns/views',
        headers=h(SVC_TOKEN3),
        json={'name': 'dup-view',
              'schema': {'type': 'struct', 'schema-id': 0,
                  'fields': [{'id': 1, 'name': 'id', 'type': 'long', 'required': True}]},
              'view-version': {'version-id': 1, 'schema-id': 0, 'timestamp-ms': 0,
                  'summary': {'engine-name': 'spark'},
                  'default-namespace': ['test-ns'],
                  'representations': [{'type': 'sql',
                      'sql': 'SELECT 1 AS id', 'dialect': 'spark'}]},
              'properties': {}})
    test('Namespace+', 'Create duplicate view', r, 409,
         'FINDING: 409 AlreadyExistsException — parallel to duplicate table')

    # Test 6: Load view that does not exist
    r = requests.get(f'{CAT}/ns2-test-catalog/namespaces/test-ns/views/nonexistent-view',
        headers=h(SVC_TOKEN3))
    test('Namespace+', 'Load non-existent view', r, 404)

finally:
    print('\n  🧹 Cleanup Group 12...')
    t = SVC_TOKEN3 or TOKEN_ROOT
    safe_delete(f'{CAT}/ns2-test-catalog/namespaces/test-ns/views/dup-view', t)
    safe_delete(f'{CAT}/ns2-test-catalog/namespaces/test-ns%1Fchild1', t)
    safe_delete(f'{CAT}/ns2-test-catalog/namespaces/test-ns%1Fchild2', t)
    safe_delete(f'{CAT}/ns2-test-catalog/namespaces/test-ns', t)
    safe_delete(f'{BASE}/principals/ns2-test-svc')
    safe_delete(f'{BASE}/principal-roles/ns2-test-pr')
    safe_delete(f'{BASE}/catalogs/ns2-test-catalog')
    print('  🧹 Done')

=== Group 12: Additional Edge Cases — Namespace & View ===
  Setup: ✅ service token obtained
  ✅ [200] Get namespace properties
  💡 Namespace properties: {'location': 's3a://data-catalog-bucket/ns2-test/test-ns/'}
  ✅ [200] Update namespace properties
       💡 POST /namespaces/{ns}/properties with updates key
  ✅ [200] List namespaces with parent filter
       💡 GET /namespaces?parent=test-ns returns only direct children
  💡 Children of test-ns: [['test-ns', 'child1'], ['test-ns', 'child2']]
  ✅ [404] Create view in non-existent namespace
       💡 FINDING: Parallel to table in non-existent namespace
  ✅ [409] Create duplicate view
       💡 FINDING: 409 AlreadyExistsException — parallel to duplicate table
  ✅ [404] Load non-existent view

  🧹 Cleanup Group 12...
  🧹 Done


## Summary Report

In [310]:
print('=== Test Summary Report ===')

df = pd.DataFrame(RESULTS)
total  = len(df)
passed = df['passed'].sum()
failed = total - passed

print(f'\nTotal: {total} | ✅ Passed: {passed} | ❌ Failed: {failed}\n')

print('By group:')
summary = df.groupby('group').agg(
    total=('passed','count'), passed=('passed','sum')
).reset_index()
summary['failed'] = summary['total'] - summary['passed']
display(summary)

if failed > 0:
    print('\n❌ Unexpected failures:')
    display(df[~df['passed']][['group','test','expected','actual','note']])

print('\n📋 Key Findings:')
findings = [
    ('Delete principal role with principals assigned', 'CASCADE — principal survives, assignment removed'),
    ('Delete catalog with namespaces inside',          'BLOCKED — 400, must empty first'),
    ('Delete namespace with tables/children inside',   'BLOCKED — 400, must be empty'),
    ('Delete table with scoped grant pointing to it',  'ALLOWED — 204, orphaned grants remain'),
    ('Create view with invalid SQL',                   'ACCEPTED — Polaris does NOT validate SQL'),
    ('Create child namespace without parent',          'BLOCKED — 404, parent must exist'),
    ('Duplicate grant / same role assignment',         '500 — duplicate key, not idempotent'),
    ('s3:// vs s3a:// at catalog creation',            'BOTH accepted — no scheme validation'),
    ('Root token on Catalog API',                      'ALLOWED — root can call Catalog API'),
    ('Delete catalog role with grants + assignments',  'CASCADE — all grants and assignments removed'),
    ('Admin rotate other principal credentials',       'BLOCKED — 403, self-only by design'),
    ('Token after principal deleted',                  'JWT stateless — verify if Polaris invalidates or accepts until expiry'),
    ('Service token on Management API',                'BLOCKED — 403, no SERVICE_MANAGE_ACCESS'),
    ('Assign same catalog role to PR twice',           '500 duplicate key — NOT idempotent (same as grant)'),
    ('Revoke grant via DELETE',                        'Supported — 200, grant removed from grant_records'),
    ('Re-grant after revocation',                      '201 — can re-grant after revoke'),
    ('List namespaces with parent filter',             '200 — returns only direct children of parent'),
    ('Create view in non-existent namespace',          '404 — same behavior as table'),
    ('Create duplicate view',                          '409 AlreadyExistsException'),
    ('Load non-existent view',                         '404'),
]
for q, a in findings:
    print(f'  • {q}')
    print(f'    → {a}')

=== Test Summary Report ===

Total: 146 | ✅ Passed: 138 | ❌ Failed: 8

By group:


,group,total,passed,failed
0,Auth,8,8,0
1,Auth+,8,8,0
2,Catalog,21,20,1
3,CatalogRole,8,8,0
4,CatalogRole+,5,5,0
5,Lifecycle,44,41,3
6,Namespace,18,15,3
7,Namespace+,6,6,0
8,Principal,9,9,0
9,PrincipalRole,7,7,0



❌ Unexpected failures:


,group,test,expected,actual,note
24,Catalog,Create catalog with s3a://,201,500,
53,Namespace,Create top-level namespace,200,500,500 but resource created
54,Namespace,Create nested namespace (parent.child),200,500,500 but resource created
55,Namespace,Create deeply nested namespace (3 levels),200,500,500 but resource created
74,Table,Create view with invalid SQL,200,500,FINDING: No SQL validation — stores definition...
90,Lifecycle,Create namespace,200,500,500 but resource created
91,Lifecycle,Create nested namespace,200,500,500 but resource created
93,Lifecycle,Create view,200,500,500 but resource created



📋 Key Findings:
  • Delete principal role with principals assigned
    → CASCADE — principal survives, assignment removed
  • Delete catalog with namespaces inside
    → BLOCKED — 400, must empty first
  • Delete namespace with tables/children inside
    → BLOCKED — 400, must be empty
  • Delete table with scoped grant pointing to it
    → ALLOWED — 204, orphaned grants remain
  • Create view with invalid SQL
    → ACCEPTED — Polaris does NOT validate SQL
  • Create child namespace without parent
    → BLOCKED — 404, parent must exist
  • Duplicate grant / same role assignment
    → 500 — duplicate key, not idempotent
  • s3:// vs s3a:// at catalog creation
    → BOTH accepted — no scheme validation
  • Root token on Catalog API
    → ALLOWED — root can call Catalog API
  • Delete catalog role with grants + assignments
    → CASCADE — all grants and assignments removed
  • Admin rotate other principal credentials
    → BLOCKED — 403, self-only by design
  • Token after principal delet